#### Topic 6: Window Functions
  - OVER() clause anatomy: PARTITION BY, ORDER BY, frame spec
  - Ranking:    RANK, DENSE_RANK, ROW_NUMBER, NTILE, PERCENT_RANK, CUME_DIST
  - Navigation: LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
  - Aggregate:  SUM, AVG, COUNT, MIN, MAX as window functions
  - Frame specs: ROWS vs RANGE, UNBOUNDED PRECEDING/FOLLOWING, CURRENT ROW

In [0]:
"""
QUICK REFERENCE:
─────────────────────────────────────────────────────────────────────────
RANKING:
  ROW_NUMBER()   → always unique (1,2,3,4)            no ties
  RANK()         → ties get same rank, gaps after      (1,1,3,4)
  DENSE_RANK()   → ties get same rank, no gaps         (1,1,2,3)
  NTILE(n)       → bucket number (1..n)
  PERCENT_RANK() → (rank-1)/(N-1), range [0,1]
  CUME_DIST()    → rows <= current / N, range (0,1]

NAVIGATION:
  LAG(col, n, default)        → n rows before (default if no row)
  LEAD(col, n, default)       → n rows after
  FIRST_VALUE(col)            → first in frame
  LAST_VALUE(col)             → last in frame (WATCH default frame!)
  NTH_VALUE(col, n)           → nth row in frame

LAST_VALUE TRAP:
  Default frame = UNBOUNDED PRECEDING to CURRENT ROW
  → Always specify: ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    to get the true last value of the partition.

FRAME DEFAULTS (when ORDER BY is present):
  RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW

WINDOW FUNCTIONS cannot be used in WHERE or HAVING clauses.
Wrap in a subquery or CTE to filter on window function results.
─────────────────────────────────────────────────────────────────────────
"""


In [0]:
# ---------------------------------------------------------------------------
# Sample data
# ---------------------------------------------------------------------------
spark.sql("""
    CREATE OR REPLACE TEMP VIEW sales AS
    SELECT * FROM VALUES
        ('2024-01', 'Engineering', 'Alice',  95000),
        ('2024-01', 'Engineering', 'Carol',  110000),
        ('2024-01', 'Engineering', 'Frank',  98000),
        ('2024-01', 'Marketing',   'Bob',    72000),
        ('2024-01', 'Marketing',   'Eve',    85000),
        ('2024-01', 'HR',          'Dave',   60000),
        ('2024-01', 'HR',          'Grace',  63000),
        ('2024-02', 'Engineering', 'Alice',  97000),
        ('2024-02', 'Engineering', 'Carol',  115000),
        ('2024-02', 'Engineering', 'Frank',  98000),
        ('2024-02', 'Marketing',   'Bob',    74000),
        ('2024-02', 'Marketing',   'Eve',    87000),
        ('2024-02', 'HR',          'Dave',   61000),
        ('2024-02', 'HR',          'Grace',  65000)
    AS t(month, department, employee, salary)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW monthly_revenue AS
    SELECT * FROM VALUES
        ('Engineering', '2024-01', 450000),
        ('Engineering', '2024-02', 520000),
        ('Engineering', '2024-03', 490000),
        ('Engineering', '2024-04', 580000),
        ('Marketing',   '2024-01', 120000),
        ('Marketing',   '2024-02', 135000),
        ('Marketing',   '2024-03', 118000),
        ('Marketing',   '2024-04', 160000),
        ('HR',          '2024-01',  40000),
        ('HR',          '2024-02',  42000),
        ('HR',          '2024-03',  41000),
        ('HR',          '2024-04',  45000)
    AS t(department, month, revenue)
""")

# ---------------------------------------------------------------------------
# 1. WINDOW FUNCTION ANATOMY
#
#  function() OVER (
#      PARTITION BY col1, col2     -- defines groups (like GROUP BY but no collapse)
#      ORDER BY col3 [ASC|DESC]    -- ordering within partition
#      ROWS|RANGE BETWEEN ...      -- frame specification
#  )
#
#  Without PARTITION BY → entire dataset is one partition
#  Without ORDER BY    → frame defaults to ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
# ---------------------------------------------------------------------------


##### 2. RANKING FUNCTIONS

In [0]:
# ---------------------------------------------------------------------------
# 2. RANKING FUNCTIONS
# ---------------------------------------------------------------------------
print("=== 2. Ranking Functions ===")
spark.sql("""
    SELECT
        month, department, employee, salary,

        -- ROW_NUMBER: unique sequential number per partition (no ties)
        ROW_NUMBER() OVER (
            PARTITION BY month, department ORDER BY salary DESC
        ) AS row_num,

        -- RANK: same rank for ties; GAPS after ties  (1,1,3)
        RANK() OVER (
            PARTITION BY month, department ORDER BY salary DESC
        ) AS rnk,

        -- DENSE_RANK: same rank for ties; NO GAPS       (1,1,2)
        DENSE_RANK() OVER (
            PARTITION BY month, department ORDER BY salary DESC
        ) AS dense_rnk,

        -- PERCENT_RANK: relative rank (0 to 1)
        -- (rank - 1) / (total_rows - 1)
        ROUND(PERCENT_RANK() OVER (
            PARTITION BY month ORDER BY salary DESC
        ), 3) AS pct_rank,

        -- CUME_DIST: cumulative distribution (0 to 1)
        -- count(rows <= current) / total_rows
        ROUND(CUME_DIST() OVER (
            PARTITION BY month ORDER BY salary DESC
        ), 3) AS cume_dist_val

    FROM sales
    ORDER BY month, department, salary DESC
""").show(20, truncate=False)


##### 3. NTILE — divide partition into N roughly equal buckets

In [0]:
# ---------------------------------------------------------------------------
# 3. NTILE — divide partition into N roughly equal buckets
# ---------------------------------------------------------------------------
print("=== 3. NTILE ===")
spark.sql("""
    SELECT
        month, employee, salary,
        NTILE(3) OVER (PARTITION BY month ORDER BY salary DESC) AS salary_tercile,
        -- 1 = top third, 2 = middle, 3 = bottom
        NTILE(4) OVER (PARTITION BY month ORDER BY salary DESC) AS salary_quartile
    FROM sales
    WHERE month = '2024-01'
    ORDER BY salary DESC
""").show()


##### 4. GET TOP-N PER GROUP 

In [0]:
# ---------------------------------------------------------------------------
# 4. GET TOP-N PER GROUP 
#    Use ROW_NUMBER or RANK in a subquery/CTE then filter
# ---------------------------------------------------------------------------
print("=== 4. Top-2 earners per department per month ===")
spark.sql("""
    WITH ranked AS (
        SELECT
            month, department, employee, salary,
            ROW_NUMBER() OVER (
                PARTITION BY month, department ORDER BY salary DESC
            ) AS rn
        FROM sales
    )
    SELECT month, department, employee, salary
    FROM ranked
    WHERE rn <= 2
    ORDER BY month, department, salary DESC
""").show(20)


##### 5. NAVIGATION FUNCTIONS — LAG, LEAD

In [0]:
# ---------------------------------------------------------------------------
# 5. NAVIGATION FUNCTIONS — LAG, LEAD
#    LAG(col, n, default)  → value n rows BEFORE current in partition
#    LEAD(col, n, default) → value n rows AFTER current in partition
# ---------------------------------------------------------------------------
print("=== 5. LAG & LEAD ===")
spark.sql("""
    SELECT
        department, month, revenue,
        LAG(revenue)           OVER (PARTITION BY department ORDER BY month) AS prev_month_rev,
        LEAD(revenue)          OVER (PARTITION BY department ORDER BY month) AS next_month_rev,
        LAG(revenue, 1, 0)     OVER (PARTITION BY department ORDER BY month) AS prev_or_zero,
        revenue -
        LAG(revenue, 1, revenue) OVER (PARTITION BY department ORDER BY month) AS mom_change,
        ROUND(
            (revenue - LAG(revenue) OVER (PARTITION BY department ORDER BY month))
            / LAG(revenue) OVER (PARTITION BY department ORDER BY month) * 100,
        1) AS mom_pct_change
    FROM monthly_revenue
    ORDER BY department, month
""").show(20, truncate=False)


##### 6. NAVIGATION FUNCTIONS — FIRST_VALUE, LAST_VALUE, NTH_VALUE

In [0]:
# ---------------------------------------------------------------------------
# 6. NAVIGATION FUNCTIONS — FIRST_VALUE, LAST_VALUE, NTH_VALUE
#
#    IMPORTANT: LAST_VALUE default frame is RANGE BETWEEN UNBOUNDED PRECEDING
#    AND CURRENT ROW — so it returns the "last seen so far", NOT partition last.
#    To get true last: use ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
# ---------------------------------------------------------------------------
print("=== 6. FIRST_VALUE, LAST_VALUE, NTH_VALUE ===")
spark.sql("""
    SELECT
        department, month, revenue,

        FIRST_VALUE(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS first_ever_rev,

        -- Default frame = CURRENT ROW only tracks up to current → "last seen so far"
        LAST_VALUE(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS last_seen_so_far,

        -- True last value in partition requires UNBOUNDED FOLLOWING
        LAST_VALUE(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS true_last_rev,

        NTH_VALUE(revenue, 2) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS second_month_rev

    FROM monthly_revenue
    ORDER BY department, month
""").show(20, truncate=False)


##### 7. AGGREGATE WINDOW FUNCTIONS

In [0]:
# ---------------------------------------------------------------------------
# 7. AGGREGATE WINDOW FUNCTIONS
#    Same functions as GROUP BY aggregates but used with OVER()
#    Key difference: rows are NOT collapsed — each row keeps its detail
# ---------------------------------------------------------------------------
print("=== 7. Aggregate Window Functions ===")
spark.sql("""
    SELECT
        month, department, employee, salary,

        -- Running total within dept per month (across all months ordered)
        SUM(salary) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,

        -- Partition total (no ORDER BY → full partition frame)
        SUM(salary) OVER (PARTITION BY month, department) AS dept_month_total,

        -- Grand total for the month (no PARTITION BY)
        SUM(salary) OVER (PARTITION BY month)             AS month_grand_total,

        -- % of dept total
        ROUND(
            salary / SUM(salary) OVER (PARTITION BY month, department) * 100, 1
        ) AS pct_of_dept,

        AVG(salary) OVER (PARTITION BY month, department) AS dept_avg,
        salary - AVG(salary) OVER (PARTITION BY month, department) AS diff_from_avg,
        COUNT(*)    OVER (PARTITION BY month, department) AS dept_headcount,
        MAX(salary) OVER (PARTITION BY month, department) AS dept_max,
        MIN(salary) OVER (PARTITION BY month, department) AS dept_min

    FROM sales
    ORDER BY month, department, salary DESC
""").show(30, truncate=False)


##### 8. FRAME SPECIFICATIONS

In [0]:
# ---------------------------------------------------------------------------
# 8. FRAME SPECIFICATIONS
#
#  ROWS BETWEEN start AND end
#    → physical rows (position-based)
#    → ROWS BETWEEN 2 PRECEDING AND CURRENT ROW = this + 2 rows before
#
#  RANGE BETWEEN start AND end
#    → logical range based on ORDER BY column values
#    → RANGE BETWEEN INTERVAL '1' DAY PRECEDING AND CURRENT ROW
#
#  Boundary keywords:
#    UNBOUNDED PRECEDING   → start of partition
#    n PRECEDING           → n rows/range units before
#    CURRENT ROW           → current row
#    n FOLLOWING           → n rows/range units after
#    UNBOUNDED FOLLOWING   → end of partition
#
#  Defaults:
#    If no frame → RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW (when ORDER BY present)
#    If no ORDER BY and no frame → ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
# ---------------------------------------------------------------------------
print("=== 8. Frame Specifications — Rolling 2-month window ===")
spark.sql("""
    SELECT
        department, month, revenue,

        -- Rolling 2-month sum (current + 1 prior) — physical rows
        SUM(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
        ) AS rolling_2m_sum,

        -- Rolling 3-month average
        ROUND(AVG(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0) AS rolling_3m_avg,

        -- Cumulative sum (running total)
        SUM(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_sum,

        -- Full partition sum (same for every row in dept)
        SUM(revenue) OVER (
            PARTITION BY department
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS partition_total,

        -- Future-looking: sum of current + next 1 row
        SUM(revenue) OVER (
            PARTITION BY department ORDER BY month
            ROWS BETWEEN CURRENT ROW AND 1 FOLLOWING
        ) AS curr_plus_next

    FROM monthly_revenue
    ORDER BY department, month
""").show(20, truncate=False)


##### 9. REAL-WORLD PATTERNS — deduplication & comparison

In [0]:
# ---------------------------------------------------------------------------
# 9. REAL-WORLD PATTERNS — deduplication & comparison
# ---------------------------------------------------------------------------
print("=== 9a. Deduplication — keep latest record per employee ===")
spark.sql("""
    WITH ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY employee ORDER BY month DESC) AS rn
        FROM sales
    )
    SELECT month, department, employee, salary
    FROM ranked
    WHERE rn = 1
    ORDER BY employee
""").show()

print("=== 9b. Salary vs department average — flag outliers ===")
spark.sql("""
    SELECT
        month,
        department,
        employee,
        salary,
        ROUND(AVG(salary) OVER (PARTITION BY month, department), 0) AS dept_avg,
        ROUND(STDDEV(salary) OVER (PARTITION BY month, department), 0) AS dept_std,
        CASE
            WHEN salary > AVG(salary) OVER (PARTITION BY month, department)
                        + STDDEV(salary) OVER (PARTITION BY month, department)
            THEN 'High outlier'
            WHEN salary < AVG(salary) OVER (PARTITION BY month, department)
                        - STDDEV(salary) OVER (PARTITION BY month, department)
            THEN 'Low outlier'
            ELSE 'Normal'
        END AS salary_status
    FROM sales
    ORDER BY month, department, salary DESC
""").show(20)

print("=== 9c. Month-over-month growth rate per department ===")
spark.sql("""
    WITH mom AS (
        SELECT
            department, month, revenue,
            LAG(revenue) OVER (PARTITION BY department ORDER BY month) AS prev_rev
        FROM monthly_revenue
    )
    SELECT
        department,
        month,
        revenue,
        prev_rev,
        revenue - prev_rev AS change,
        CASE
            WHEN prev_rev IS NULL THEN NULL
            ELSE ROUND((revenue - prev_rev) / prev_rev * 100, 1)
        END AS pct_growth
    FROM mom
    ORDER BY department, month
""").show(20)


In [0]:
spark.stop()